# Deep Learning for Image Analysis
### YOLO Object Detection on Pascal VOC 2012

**Authors:** Bo Fu, Yehoshua Perez Condori  
**Programme:** MSc Artificial Intelligence  
**Institution:** City St George's, University of London  
**Module:** INM705 Deep Learning for Image Analysis  
**Module Leader:** Dr Riad Ibadulla  

---

### Project Overview
Implementation of YOLOv1-style object detection using VGG16 backbone trained on Pascal VOC 2012 dataset with 20 object categories.

---

### Links
- **GitHub:** https://github.com/BoFu001/YOLO-object-detection
- **Colab Notebook:** https://drive.google.com/file/d/182m9Fadqzu_SJAwi9HJrPFqUUiMgEdhU/view?usp=sharing
- **Kaggle Notebook:** https://www.kaggle.com/code/bofu001/yolo-object-detection
- **Kaggle Dataset:** https://www.kaggle.com/datasets/huanghanchina/pascal-voc-2012
- **Wandb:** https://wandb.ai/bofu001-/YOLO-VOC2012

In [2]:
import os
print('Current working directory:', os.getcwd())

Current working directory: /teamspace/studios/this_studio/YOLO-object-detection


In [3]:
import os
print('Drive mounted:', os.path.exists('./data/'))
if os.path.exists('./data/'):
    print('Contents of MyDrive:')
    print(os.listdir('./data/')[:10])

# Let's also check if Colab Notebooks exists
if os.path.exists('./data/Colab Notebooks/'):
    print('\nContents of Colab Notebooks:')
    print(os.listdir('./data/Colab Notebooks/')[:10])

Drive mounted: False


In [12]:
import sys

PROJECT_ROOT = "/teamspace/studios/this_studio/YOLO-object-detection"
modules_dir = f"{PROJECT_ROOT}/modules"

In [13]:

print(os.path.exists(f"{PROJECT_ROOT}/my_config.py"))
print(os.path.exists(f"{PROJECT_ROOT}/modules"))



True
True


In [14]:
import os

if os.getenv("WANDB_API_KEY"):
    os.environ.pop("WANDB_MODE", None)
    print("W&B enabled from Lightning secret.")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("WANDB_API_KEY not found. W&B disabled.")

W&B enabled from Lightning secret.


In [28]:
from pathlib import Path
import os

VOC_ROOT = Path("/teamspace/lightning_storage/data/VOC2012")
ZIP_PATH = Path("/teamspace/lightning_storage/data/pascal-voc-2012.zip")
KAGGLE_JSON = Path("/teamspace/studios/this_studio/YOLO-object-detection/kaggle.json")

if not KAGGLE_JSON.exists():
    raise FileNotFoundError(f"Missing kaggle.json at: {KAGGLE_JSON}")

os.environ["KAGGLE_CONFIG_DIR"] = str(KAGGLE_JSON.parent)
os.chmod(KAGGLE_JSON, 0o600)

print("KAGGLE_CONFIG_DIR =", os.environ["KAGGLE_CONFIG_DIR"])
print("Using kaggle.json from:", KAGGLE_JSON)

KAGGLE_CONFIG_DIR = /teamspace/studios/this_studio/YOLO-object-detection
Using kaggle.json from: /teamspace/studios/this_studio/YOLO-object-detection/kaggle.json


In [30]:
import zipfile
import subprocess

VOC_ROOT.parent.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    print("Downloading Pascal VOC 2012 from Kaggle...")
    subprocess.run([
        "kaggle", "datasets", "download",
        "-d", "huanghanchina/pascal-voc-2012",
        "-p", str(ZIP_PATH.parent)
    ], check=True)
else:
    print("Zip already exists:", ZIP_PATH)

if not VOC_ROOT.exists():
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(VOC_ROOT.parent)
else:
    print("Dataset already extracted at:", VOC_ROOT)

print("Done.")

Dataset URL: https://www.kaggle.com/datasets/huanghanchina/pascal-voc-2012
License(s): DbCL-1.0


100%|██████████| 3.63G/3.63G [00:17<00:00, 228MB/s] 



Extracting dataset...
Done.


In [23]:
import random
import numpy as np
import torch
import io
import os
import re
import json
import wandb
import sys
import pandas as pd
from IPython.display import display


In [24]:
from my_config import SEED, CKPT_DIR, DEVICE, IMG_DIR, ANN_DIR, CLASSES, NUM_WORKERS

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [25]:
# custom modules
from modules.Dataset import get_dataloaders
from modules.Train import train
from modules.TrainFinetune import train_finetune
from modules.TrainFinetuneLayerwise import train_finetune_layerwise
from modules.Models.YOLOv1 import YOLOv1
from modules.Models.YOLOv1Dropout import YOLOv1Dropout
from modules.Models.YOLOv1Finetune import YOLOv1Finetune
from modules.Evaluation import evaluate
from modules.Inference import inference
from contextlib import contextmanager, redirect_stdout

Class and Method Declaration

In [ ]:

@contextmanager
def reuse_existing_wandb_run():
    """Reuse the active sweep run inside helper functions that may call wandb.init/finish."""
    original_init = wandb.init
    original_finish = wandb.finish

    def _reuse_init(*args, **kwargs):
        return wandb.run if wandb.run is not None else original_init(*args, **kwargs)

    def _noop_finish(*args, **kwargs):
        return None

    wandb.init = _reuse_init
    wandb.finish = _noop_finish
    try:
        yield
    finally:
        wandb.init = original_init
        wandb.finish = original_finish

In [ ]:

def parse_map_metrics(eval_text):
    map50_match = re.search(r"mAP@0\.50:\s*([0-9]*\.?[0-9]+)", eval_text)
    map5095_match = re.search(r"mAP@0\.50:0\.95:\s*([0-9]*\.?[0-9]+)", eval_text)

    map50 = float(map50_match.group(1)) if map50_match else None
    map5095 = float(map5095_match.group(1)) if map5095_match else None
    return {
        "val/mAP": map50,
        "val/mAP_50_95": map5095,
    }

In [ ]:

def set_dropout_p(model, dropout_p):
    """Update all Dropout layers in-place. Safe even if the model has no dropout layers."""
    dropout_layers = 0
    for module in model.modules():
        if isinstance(module, torch.nn.Dropout):
            module.p = float(dropout_p)
            dropout_layers += 1
    print(f"Updated {dropout_layers} dropout layer(s) to p={float(dropout_p):.3f}")
    return dropout_layers

In [ ]:
BEST_SWEEP_CKPT = None
BEST_SWEEP_RUN = None
BEST_SWEEP_MAP50 = float("-inf")
BEST_SWEEP_SUMMARY = None

In [ ]:
def build_sweep_model(dropout_p):
    model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)
    set_dropout_p(model, dropout_p)

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = torch.nn.DataParallel(model)

    raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model
    return model, raw_model

In [ ]:
def train_sweep():
    global BEST_SWEEP_CKPT, BEST_SWEEP_RUN, BEST_SWEEP_MAP50, BEST_SWEEP_SUMMARY

    run = wandb.init(project=SWEEP_PROJECT)
    config = wandb.config

    run_name = f"sweep_{run.id}"
    wandb.run.name = run_name
    print(f"Starting sweep run: {run_name}")

    # fresh loaders for each run
    train_loader, val_loader, _ = get_dataloaders(
        int(config.BATCH_SIZE),
        S, B, C,
        augment=bool(config.AUGMENT)
    )

    model, raw_model = build_sweep_model(config.DROPOUT_P)

    ckpt_path = os.path.join(CKPT_DIR, f"{run_name}.pth")

    with reuse_existing_wandb_run():
        raw_model = train_finetune_layerwise(
            model=model,
            raw_model=raw_model,
            train_loader=train_loader,
            val_loader=val_loader,
            S=S, B=B, C=C,
            BATCH_SIZE=int(config.BATCH_SIZE),
            EPOCHS=int(config.EPOCHS),
            LR_HEAD=float(config.LR_HEAD),
            LR_BACKBONE=float(config.LR_BACKBONE),
            WEIGHT_DECAY=float(config.WEIGHT_DECAY),
            LAMBDA_BOX=float(config.LAMBDA_BOX),
            LAMBDA_NOOBJ=float(config.LAMBDA_NOOBJ),
            RUN_NAME=run_name
        )

    torch.save(raw_model.state_dict(), ckpt_path)
    print(f"Sweep weights saved: {ckpt_path}")

    raw_model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    metrics, eval_text = evaluate_with_capture(
        model=model,
        loader=val_loader,
        conf_thresh=float(config.CONF_THRESH),
        iou_thresh=NMS_IOU_THRESH
    )

    summary = {
        "val/mAP": metrics["val/mAP"],
        "val/mAP_50_95": metrics["val/mAP_50_95"],
        "ckpt_path": ckpt_path,
        "dropout_p": float(config.DROPOUT_P),
        "conf_thresh": float(config.CONF_THRESH),
        "augment": bool(config.AUGMENT),
    }

    wandb.log(summary)
    wandb.run.summary["ckpt_path"] = ckpt_path
    wandb.run.summary["eval_stdout"] = eval_text

    if metrics["val/mAP"] is not None and metrics["val/mAP"] > BEST_SWEEP_MAP50:
        BEST_SWEEP_MAP50 = metrics["val/mAP"]
        BEST_SWEEP_CKPT = ckpt_path
        BEST_SWEEP_RUN = run_name
        BEST_SWEEP_SUMMARY = {
            "run_name": run_name,
            "ckpt_path": ckpt_path,
            "val/mAP": metrics["val/mAP"],
            "val/mAP_50_95": metrics["val/mAP_50_95"],
            "config": dict(wandb.config),
        }

        with open(os.path.join(CKPT_DIR, "best_sweep_summary.json"), "w") as f:
            json.dump(BEST_SWEEP_SUMMARY, f, indent=2)

        print("New best sweep run found:")
        print(json.dumps(BEST_SWEEP_SUMMARY, indent=2))

    wandb.finish()

In [ ]:

def evaluate_with_capture(model, loader, conf_thresh, iou_thresh):
    """Run evaluate() on the validation loader and parse printed mAP values."""
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        _ = evaluate(
            model=model,
            test_loader=loader,
            S=S, B=B, C=C,
            conf_thresh=float(conf_thresh),
            iou_thresh=iou_thresh
        )
    eval_text = buffer.getvalue()
    print(eval_text)
    metrics = parse_map_metrics(eval_text)
    return metrics, eval_text

In [ ]:
# YOLO parameters
S = 7
B = 2
C = 20

# training parameters
BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4

# loss weights
LAMBDA_BOX = 5.0
LAMBDA_NOOBJ = 0.5

# inference thresholds
CONF_THRESH = 0.30
NMS_IOU_THRESH = 0.45

In [ ]:
# create all data loaders
train_loader, val_loader, test_loader = get_dataloaders(BATCH_SIZE, S, B, C)

#### Experiment 1 - Baseline
* Model: YOLOv1 (frozen VGG16 backbone)
* LR: 1e-3
* EPOCHS: 5
* Goal: verify model can learn and observe initial loss trend

In [ ]:
RUN_NAME = "exp1_YOLOv1_lr1e-3"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS       = 5
LR           = 1e-3

In [ ]:
# create model
model = YOLOv1(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)
else :
    print("not using GPUs")

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

Training: Experiment 1

In [ ]:
# training
raw_model = train(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)

# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

Evaluation: Experiment 1

In [ ]:
# evaluation

# load trained weights before evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))

results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 2 - Lower Learning Rate
* Model: YOLOv1 (frozen VGG16 backbone)
* LR: 1e-3 → 1e-4
* EPOCHS: 20
* Goal: reduce overfitting seen in Experiment 1

In [ ]:
RUN_NAME   = "exp2_YOLOv1_lr1e-4"
CKPT_PATH  = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS       = 20
LR           = 1e-4

In [ ]:
# create model
model = YOLOv1(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

Training : Experiment 2

In [ ]:
# training
raw_model = train(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)

# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

Evaluation: Experiment 2

In [ ]:
# evaluation

# load trained weights before evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))

results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 3 - Dropout Regularisation
* Model: YOLOv1Dropout (frozen VGG16 backbone)
* LR: 1e-4
* EPOCHS: 20
* Dropout: p=0.5 added in head
* Goal: further reduce overfitting with dropout regularisation

In [ ]:
RUN_NAME  = "exp3_Dropout_lr1e-4"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR = 1e-4

In [ ]:
# create model
model = YOLOv1Dropout(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

Training: Experiment 3

In [ ]:

# training
raw_model = train(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS, LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")


Evaluation: Experiment 3

In [ ]:
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 4 - VGG16 Fine-tuning with Early Stopping
* Model: YOLOv1Finetune (last 2 conv layers of VGG16 unfrozen)
* LR: 1e-4
* EPOCHS: 20 (Early Stopping: patience=5)
* Dropout: p=0.5 in head
* Goal: improve feature extraction by fine-tuning backbone on VOC dataset

In [ ]:
RUN_NAME  = "exp4_Finetune_lr1e-4"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR = 1e-4

In [ ]:
# create model
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model

Training: Experiment 4

In [ ]:
# training
raw_model = train_finetune(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

Evaluation: Experiment 4

In [ ]:
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 5 - Layer-wise Learning Rate
* Model: YOLOv1Finetune (last 2 conv layers of VGG16 unfrozen)
* LR head: 1e-4
* LR backbone: 1e-5
* EPOCHS: 20 (Early Stopping: patience=5)
* Dropout: p=0.5 in head
* Goal: more stable fine-tuning with smaller backbone LR

In [ ]:
RUN_NAME  = "exp5_Finetune_lrH1e-4_lrB1e-5"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR_HEAD=1e-4
LR_BACKBONE=1e-5

In [ ]:
# create model
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model


Training: Experiment 5

In [ ]:
# training
raw_model = train_finetune_layerwise(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR_HEAD=LR_HEAD,
    LR_BACKBONE=LR_BACKBONE,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")

Evaluation: Experiment 5

In [ ]:
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 6 - Layer-wise LR Tuning
* Model: YOLOv1Finetune (last 2 conv layers of VGG16 unfrozen)
* LR head: 1e-4
* LR backbone: 5e-5 (increased from exp5)
* EPOCHS: 20 (Early Stopping: patience=5)
* Dropout: p=0.5 in head
* Goal: find better backbone LR between exp4 (1e-4) and exp5 (1e-5)

In [ ]:
RUN_NAME = "exp6_Finetune_lrH1e-4_lrB5e-5"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR_HEAD=1e-4
LR_BACKBONE= 5e-5

In [ ]:
# create model
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model


Training: Experiment 6

In [ ]:
# training
raw_model = train_finetune_layerwise(
    model = model,
    raw_model = raw_model,
    train_loader = train_loader,
    val_loader = val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE = BATCH_SIZE,
    EPOCHS = EPOCHS,
    LR_HEAD = LR_HEAD,
    LR_BACKBONE = LR_BACKBONE,
    WEIGHT_DECAY = WEIGHT_DECAY,
    LAMBDA_BOX = LAMBDA_BOX,
    LAMBDA_NOOBJ = LAMBDA_NOOBJ,
    RUN_NAME = RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")


Evluation: Experiment 6

In [ ]:
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 7 - Data Augmentation
* Model: YOLOv1Finetune (last 2 conv layers of VGG16 unfrozen)
* LR: 1e-4
* EPOCHS: 20 (Early Stopping: patience=5)
* Dropout: p=0.5 in head
* Augmentation: ColorJitter (brightness, contrast, saturation, hue)
* Goal: reduce overfitting with colour augmentation on training set

In [ ]:
RUN_NAME = "exp7_Finetune_Aug_lr1e-4"
CKPT_PATH = f"{CKPT_DIR}/{RUN_NAME}.pth"
EPOCHS = 20
LR = 1e-4

In [ ]:
# create data loaders with augmentation
train_loader, val_loader, test_loader = get_dataloaders(BATCH_SIZE, S, B, C, augment=True)

In [ ]:
# create model
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)

# multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

# unwrap DataParallel
raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model


Training: Experiment 7

In [ ]:
# training
raw_model = train_finetune(
    model=model,
    raw_model=raw_model,
    train_loader=train_loader,
    val_loader=val_loader,
    S=S, B=B, C=C,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    LAMBDA_BOX=LAMBDA_BOX,
    LAMBDA_NOOBJ=LAMBDA_NOOBJ,
    RUN_NAME=RUN_NAME
)
# save trained weights
torch.save(raw_model.state_dict(), CKPT_PATH)
print(f"trained weights saved: {RUN_NAME}.pth")


Evaluation: Experiment 7

In [ ]:
# evaluation
raw_model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
results = evaluate(
    model=model,
    test_loader=test_loader,
    S=S, B=B, C=C,
    conf_thresh=CONF_THRESH,
    iou_thresh=NMS_IOU_THRESH
)

#### Experiment 8 - Automated Hyperparameter Tuning with WandB Sweeps
This section adds a **Bayesian WandB Sweep** on top of the strongest manual setup:
* Model: `YOLOv1Finetune`
* Optimiser style: **layer-wise learning rates**
* Sweep metric: **validation mAP@0.50**
* Search space: `LR_HEAD`, `LR_BACKBONE`, `WEIGHT_DECAY`, `LAMBDA_BOX`, `LAMBDA_NOOBJ`, `DROPOUT_P`, `CONF_THRESH`

Notes:
* This reuses the existing `train_finetune_layerwise()` training function.
* A small WandB patch is included so the sweep run is reused even if your training helpers already call `wandb.init()` or `wandb.finish()`.
* `DROPOUT_P` is applied by updating any `torch.nn.Dropout` layers found in the model.

Weights and Biases Sweep: Results

In [ ]:

SWEEP_PROJECT = "yolo-object-detection"
# How many runs (trials) the agent will execute. Increase for better search, decrease for time/compute limits.
SWEEP_COUNT = 20  # reduce this if Colab time is tight

sweep_config = {
    # Bayesian optimisation over the parameters below
    "method": "bayes",

    # The scalar to maximise across sweep trials
    "metric": {"name": "val/mAP", "goal": "maximize"},

    "parameters": {
        # Fixed knobs (kept constant across all trials)
        "EPOCHS": {"value": 20},
        "BATCH_SIZE": {"value": BATCH_SIZE},
        "AUGMENT": {"value": False},

        # Learning rates: search on a log scale (LRs typically vary by orders of magnitude)
        "LR_HEAD": {
            "min": 1e-5,
            "max": 1e-3,
            "distribution": "log_uniform_values",
        },
        "LR_BACKBONE": {
            "min": 1e-6,
            "max": 1e-4,
            "distribution": "log_uniform_values",
        },

        # Weight decay: also varies best on a log scale
        "WEIGHT_DECAY": {
            "min": 1e-5,
            "max": 1e-3,
            "distribution": "log_uniform_values",
        },

        # YOLO loss weights: linear ranges are OK (these are already in human-scale ranges)
        "LAMBDA_BOX": {
            "min": 2.0,
            "max": 10.0,
        },
        "LAMBDA_NOOBJ": {
            "min": 0.1,
            "max": 1.0,
        },

        # Dropout probability applied to any `torch.nn.Dropout` layers found in the model
        # (If the model has no Dropout layers, this won't change anything.)
        "DROPOUT_P": {
            "min": 0.3,
            "max": 0.7,
        },

        # Confidence threshold used during evaluation to filter predictions.
        # NOTE: This is an *evaluation-time* knob, not training-time. Optimising it can inflate mAP by tuning the
        # decision threshold; keep it fixed if you want strict apples-to-apples model comparisons.
        "CONF_THRESH": {
            "min": 0.2,
            "max": 0.5,
        },
    },
}

print(json.dumps(sweep_config, indent=2))

In [ ]:
sweep_id = wandb.sweep(sweep_config, project=SWEEP_PROJECT)
print(f"Sweep ID: {sweep_id}")
wandb.agent(sweep_id, function=train_sweep, count=SWEEP_COUNT)

print("\nBest sweep summary:")
print(json.dumps(BEST_SWEEP_SUMMARY, indent=2) if BEST_SWEEP_SUMMARY else "No successful sweep result captured.")

In [ ]:
import pandas as pd
import wandb

api = wandb.Api()

# replace with your actual values
sweep = api.sweep("y-benjamin_pc-city-st-george-s-university-of-london/yolo-object-detection/8ljoiuem")

rows = []
for run in sweep.runs:
    cfg = {k: v for k, v in run.config.items() if not k.startswith("_")}
    summ = dict(run.summary)

    rows.append({
        "run_name": run.name,
        "state": run.state,
        "val/mAP": summ.get("val/mAP"),
        "val/mAP_50_95": summ.get("val/mAP_50_95"),
        "train/loss": summ.get("train/loss"),
        "val/loss": summ.get("val/loss"),
        "precision": summ.get("val/precision"),
        "recall": summ.get("val/recall"),
        "LR_HEAD": cfg.get("LR_HEAD"),
        "LR_BACKBONE": cfg.get("LR_BACKBONE"),
        "LAMBDA_BOX": cfg.get("LAMBDA_BOX"),
        "LAMBDA_NOOBJ": cfg.get("LAMBDA_NOOBJ"),
        "WEIGHT_DECAY": cfg.get("WEIGHT_DECAY"),
        "DROPOUT_P": cfg.get("DROPOUT_P"),
        "CONF_THRESH": cfg.get("CONF_THRESH"),
        "BATCH_SIZE": cfg.get("BATCH_SIZE"),
        "AUGMENT": cfg.get("AUGMENT"),
    })

df = pd.DataFrame(rows)

# best by val/mAP
df_map = df.sort_values("val/mAP", ascending=False)

# best by stricter localisation metric
df_map5095 = df.sort_values("val/mAP_50_95", ascending=False)

print("Top 10 by val/mAP")
print(df_map.head(10).to_string(index=False))

print("\nTop 10 by val/mAP_50_95")
print(df_map5095.head(10).to_string(index=False))

df.to_csv("sweep_results_full.csv", index=False)

In [ ]:
import wandb
import pandas as pd

ENTITY = "y-benjamin_pc-city-st-george-s-university-of-london"
PROJECT = "yolo-object-detection"
TOP_K = 10

api = wandb.Api()
runs = api.runs(f"{ENTITY}/{PROJECT}")

rows = []

for run in runs:
    summary = run.summary or {}
    config = run.config or {}

    rows.append({
        "run_name": run.name,
        "state": run.state,
        "val/mAP": summary.get("val/mAP"),
        "val/mAP_50_95": summary.get("val/mAP_50_95"),
        "train/loss": summary.get("train/loss"),
        "val/loss": summary.get("val/loss"),
        "precision": summary.get("precision"),
        "recall": summary.get("recall"),
        "LR_HEAD": config.get("LR_HEAD"),
        "LR_BACKBONE": config.get("LR_BACKBONE"),
        "LAMBDA_BOX": config.get("LAMBDA_BOX"),
        "LAMBDA_NOOBJ": config.get("LAMBDA_NOOBJ"),
        "WEIGHT_DECAY": config.get("WEIGHT_DECAY"),
        "DROPOUT_P": config.get("DROPOUT_P"),
        "CONF_THRESH": config.get("CONF_THRESH"),
        "BATCH_SIZE": config.get("BATCH_SIZE"),
        "AUGMENT": config.get("AUGMENT"),
    })

df = pd.DataFrame(rows)

# top runs by val/mAP
top_map = (
    df.dropna(subset=["val/mAP"])
      .sort_values("val/mAP", ascending=False)
      .head(TOP_K)
)

print(f"Top {TOP_K} by val/mAP")
print(top_map.to_string(index=False))

# top runs by val/mAP_50_95
top_map5095 = (
    df.dropna(subset=["val/mAP_50_95"])
      .sort_values("val/mAP_50_95", ascending=False)
      .head(TOP_K)
)

print("\n" + "="*100 + "\n")
print(f"Top {TOP_K} by val/mAP_50_95")
print(top_map5095.to_string(index=False))

In [ ]:
ENTITY = "y-benjamin_pc-city-st-george-s-university-of-london"
PROJECT = "yolo-object-detection"

# folder where outputs will be saved
OUTPUT_DIR = Path("./data/Colab Notebooks/Education/Deep Learning for Image Analysis/YOLO-object-detection/graphs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# metrics to export if present
LOSS_METRICS = [
    "train_loss",
    "val_loss",
    "box_loss",
    "obj_loss",
    "noobj_loss",
    "cls_loss",
]

# if you only want runs whose names contain something, set it here
# example: RUN_NAME_FILTER = "exp"
RUN_NAME_FILTER = None

# if True, save raw CSV history for each run
SAVE_CSV = True


def safe_filename(text: str) -> str:
    """Make a filename safe for Windows/macOS/Linux."""
    bad = '<>:"/\\|?*'
    for ch in bad:
        text = text.replace(ch, "_")
    return text.strip().replace(" ", "_")


def get_epoch_or_step(df: pd.DataFrame) -> pd.Series:
    """Use epoch if present, else _step, else dataframe index."""
    if "epoch" in df.columns:
        return df["epoch"]
    if "_step" in df.columns:
        return df["_step"]
    return pd.Series(df.index, index=df.index)


def export_run_history(run, output_dir: Path) -> pd.DataFrame | None:
    """Download one run's history and optionally save CSV."""
    try:
        df = run.history(samples=100000)
    except Exception as e:
        print(f"[SKIP] Could not load history for {run.name}: {e}")
        return None

    if df is None or df.empty:
        print(f"[SKIP] No history for {run.name}")
        return None

    run_slug = safe_filename(f"{run.name}_{run.id}")
    run_dir = output_dir / run_slug
    run_dir.mkdir(parents=True, exist_ok=True)

    if SAVE_CSV:
        csv_path = run_dir / "history.csv"
        df.to_csv(csv_path, index=False)

    return df


def plot_individual_losses(run, df: pd.DataFrame, output_dir: Path) -> None:
    """Save one PNG per loss metric if present."""
    run_slug = safe_filename(f"{run.name}_{run.id}")
    run_dir = output_dir / run_slug
    x = get_epoch_or_step(df)

    for metric in LOSS_METRICS:
        if metric not in df.columns:
            continue

        metric_df = pd.DataFrame({"x": x, "y": df[metric]}).dropna()
        if metric_df.empty:
            continue

        plt.figure(figsize=(8, 5))
        plt.plot(metric_df["x"], metric_df["y"])
        plt.xlabel("Epoch" if "epoch" in df.columns else "Step")
        plt.ylabel(metric)
        plt.title(f"{run.name} - {metric}")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(run_dir / f"{metric}.png", dpi=200)
        plt.close()


def plot_train_vs_val(run, df: pd.DataFrame, output_dir: Path) -> None:
    """Save a combined train vs val loss graph if both exist."""
    if "train_loss" not in df.columns or "val_loss" not in df.columns:
        return

    run_slug = safe_filename(f"{run.name}_{run.id}")
    run_dir = output_dir / run_slug
    x = get_epoch_or_step(df)

    pair_df = pd.DataFrame({
        "x": x,
        "train_loss": df["train_loss"],
        "val_loss": df["val_loss"],
    }).dropna(how="all")

    if pair_df.empty:
        return

    plt.figure(figsize=(8, 5))
    if pair_df["train_loss"].notna().any():
        plt.plot(pair_df["x"], pair_df["train_loss"], label="train_loss")
    if pair_df["val_loss"].notna().any():
        plt.plot(pair_df["x"], pair_df["val_loss"], label="val_loss")

    plt.xlabel("Epoch" if "epoch" in df.columns else "Step")
    plt.ylabel("Loss")
    plt.title(f"{run.name} - Train vs Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(run_dir / "train_vs_val_loss.png", dpi=200)
    plt.close()


def build_summary_row(run, df: pd.DataFrame) -> dict:
    """Create a summary row for quick comparison across runs."""
    row = {
        "run_name": run.name,
        "run_id": run.id,
        "state": getattr(run, "state", None),
        "url": getattr(run, "url", None),
    }

    summary = getattr(run, "summary", {}) or {}

    # W&B summary keys vary depending on logging style
    candidate_keys = [
        "val_loss",
        "train_loss",
        "box_loss",
        "obj_loss",
        "noobj_loss",
        "cls_loss",
        "val/mAP",
        "val/mAP50",
        "metrics/mAP50",
        "mAP@0.50",
        "mAP50",
    ]

    for key in candidate_keys:
        row[key] = summary.get(key, None)

    # fallback: final history values if summary is missing
    for metric in LOSS_METRICS:
        if row.get(metric) is None and metric in df.columns:
            non_null = df[metric].dropna()
            row[metric] = non_null.iloc[-1] if not non_null.empty else None

    return row


def plot_compare_metric_across_runs(summary_df: pd.DataFrame, metric: str, output_dir: Path) -> None:
    """Plot one bar chart across runs for a given metric."""
    if metric not in summary_df.columns:
        return

    temp = summary_df[["run_name", metric]].dropna()
    if temp.empty:
        return

    # sort so the chart is easier to read
    temp = temp.sort_values(by=metric, ascending=True)

    plt.figure(figsize=(10, max(5, len(temp) * 0.35)))
    plt.barh(temp["run_name"], temp[metric])
    plt.xlabel(metric)
    plt.ylabel("Run")
    plt.title(f"{metric} across runs")
    plt.tight_layout()
    plt.savefig(output_dir / f"compare_{safe_filename(metric)}.png", dpi=200)
    plt.close()


def main():
    api = wandb.Api()
    runs = api.runs(f"{ENTITY}/{PROJECT}")

    summary_rows = []
    processed = 0

    for run in runs:
        if RUN_NAME_FILTER and RUN_NAME_FILTER.lower() not in (run.name or "").lower():
            continue

        print(f"[INFO] Processing run: {run.name} ({run.id})")
        df = export_run_history(run, OUTPUT_DIR)
        if df is None:
            continue

        plot_individual_losses(run, df, OUTPUT_DIR)
        plot_train_vs_val(run, df, OUTPUT_DIR)

        summary_rows.append(build_summary_row(run, df))
        processed += 1

    if not summary_rows:
        print("[DONE] No runs matched.")
        return

    summary_df = pd.DataFrame(summary_rows)
    summary_path = OUTPUT_DIR / "run_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    # comparison charts across runs
    for metric in ["val_loss", "train_loss", "val/mAP", "val/mAP50", "mAP@0.50", "mAP50"]:
        plot_compare_metric_across_runs(summary_df, metric, OUTPUT_DIR)

    print(f"[DONE] Processed {processed} runs.")
    print(f"[DONE] Output saved to: {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    main()

Experiment 9

In [31]:

FINAL_RETRAIN_SEEDS = [SEED, SEED + 1, SEED + 2]
FINAL_RESULTS_CSV = os.path.join(CKPT_DIR, "exp9_final_retrain_results.csv")
FINAL_SUMMARY_JSON = os.path.join(CKPT_DIR, "exp9_final_retrain_summary.json")

BEST_FINAL_CKPT = None
BEST_FINAL_RUN_NAME = None
BEST_FINAL_SUMMARY = None
FINAL_RETRAIN_RESULTS = None
FINAL_RETRAIN_DF = None
COMPARISON_DF = None

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def load_best_sweep_summary_from_disk_or_memory():
    if BEST_SWEEP_SUMMARY is not None:
        return BEST_SWEEP_SUMMARY

    summary_path = os.path.join(CKPT_DIR, "best_sweep_summary.json")
    if not os.path.exists(summary_path):
        raise FileNotFoundError(
            "best_sweep_summary.json was not found. Run Experiment 8 first, then rerun this section."
        )

    with open(summary_path, "r") as f:
        return json.load(f)

TARGET_SWEEP_RUN = ""   # put your actual run name here

sweep_results_path = os.path.join(CKPT_DIR, "sweep_results.csv")
sweep_df = pd.read_csv(sweep_results_path)

target_row = sweep_df[sweep_df["run_name"] == TARGET_SWEEP_RUN]

if target_row.empty:
    raise ValueError(f"Run {TARGET_SWEEP_RUN} not found in {sweep_results_path}")

FINAL_SWEEP_CONFIG = json.loads(target_row.iloc[0]["config_json"])


FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/sweep_results.csv'

In [ ]:

FINAL_RETRAIN_DF = pd.DataFrame(FINAL_RETRAIN_RESULTS)
display(FINAL_RETRAIN_DF)

if FINAL_RETRAIN_DF.empty:
    raise RuntimeError("No final retraining results were collected.")

FINAL_RETRAIN_DF.to_csv(FINAL_RESULTS_CSV, index=False)

best_final_idx = FINAL_RETRAIN_DF["val/mAP"].astype(float).idxmax()
best_final_row = FINAL_RETRAIN_DF.loc[best_final_idx].to_dict()

BEST_FINAL_CKPT = best_final_row["ckpt_path"]
BEST_FINAL_RUN_NAME = best_final_row["run_name"]
BEST_FINAL_SUMMARY = {
    "best_final_run_name": BEST_FINAL_RUN_NAME,
    "best_final_ckpt": BEST_FINAL_CKPT,
    "best_final_seed": int(best_final_row["seed"]),
    "best_final_val_mAP": float(best_final_row["val/mAP"]),
    "best_final_test_mAP": float(best_final_row["test/mAP"]),
    "mean_val_mAP": float(FINAL_RETRAIN_DF["val/mAP"].astype(float).mean()),
    "std_val_mAP": float(FINAL_RETRAIN_DF["val/mAP"].astype(float).std(ddof=0)),
    "mean_test_mAP": float(FINAL_RETRAIN_DF["test/mAP"].astype(float).mean()),
    "std_test_mAP": float(FINAL_RETRAIN_DF["test/mAP"].astype(float).std(ddof=0)),
    "mean_val_mAP_50_95": float(FINAL_RETRAIN_DF["val/mAP_50_95"].astype(float).mean()),
    "std_val_mAP_50_95": float(FINAL_RETRAIN_DF["val/mAP_50_95"].astype(float).std(ddof=0)),
    "mean_test_mAP_50_95": float(FINAL_RETRAIN_DF["test/mAP_50_95"].astype(float).mean()),
    "std_test_mAP_50_95": float(FINAL_RETRAIN_DF["test/mAP_50_95"].astype(float).std(ddof=0)),
    "final_retrain_seeds": FINAL_RETRAIN_SEEDS,
    "source_sweep_run_name": BEST_SWEEP_SUMMARY["run_name"],
    "source_sweep_ckpt": BEST_SWEEP_SUMMARY["ckpt_path"],
    "source_sweep_config": FINAL_SWEEP_CONFIG,
}

with open(FINAL_SUMMARY_JSON, "w") as f:
    json.dump(BEST_FINAL_SUMMARY, f, indent=2)

print("Final retraining summary:")
print(json.dumps(BEST_FINAL_SUMMARY, indent=2))


In [ ]:

# Fair comparison on the same evaluation threshold.
comparison_conf_thresh = float(FINAL_SWEEP_CONFIG.get("CONF_THRESH", CONF_THRESH))

# Evaluation loaders do not need augmentation.
eval_train_loader, eval_val_loader, eval_test_loader = get_dataloaders(BATCH_SIZE, S, B, C)

def evaluate_checkpoint_path(label, ckpt_path, loader_val, loader_test, conf_thresh):
    if ckpt_path is None or not os.path.exists(ckpt_path):
        print(f"Skipping {label}: checkpoint not found -> {ckpt_path}")
        return {
            "model_label": label,
            "ckpt_path": ckpt_path,
            "val/mAP": None,
            "val/mAP_50_95": None,
            "test/mAP": None,
            "test/mAP_50_95": None,
        }

    model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    print("=" * 100)
    print(f"Comparison evaluation: {label}")
    print(f"Checkpoint: {ckpt_path}")

    val_metrics, _ = evaluate_with_capture(
        model=model,
        loader=loader_val,
        conf_thresh=conf_thresh,
        iou_thresh=NMS_IOU_THRESH
    )

    test_metrics, _ = evaluate_with_capture(
        model=model,
        loader=loader_test,
        conf_thresh=conf_thresh,
        iou_thresh=NMS_IOU_THRESH
    )

    return {
        "model_label": label,
        "ckpt_path": ckpt_path,
        "val/mAP": val_metrics["val/mAP"],
        "val/mAP_50_95": val_metrics["val/mAP_50_95"],
        "test/mAP": test_metrics["val/mAP"],
        "test/mAP_50_95": test_metrics["val/mAP_50_95"],
    }

comparison_rows = [
    evaluate_checkpoint_path(
        label="Best manual experiment (exp6)",
        ckpt_path=BEST_MANUAL_CKPT,
        loader_val=eval_val_loader,
        loader_test=eval_test_loader,
        conf_thresh=comparison_conf_thresh,
    ),
    evaluate_checkpoint_path(
        label="Best Bayesian sweep trial",
        ckpt_path=BEST_SWEEP_SUMMARY["ckpt_path"],
        loader_val=eval_val_loader,
        loader_test=eval_test_loader,
        conf_thresh=comparison_conf_thresh,
    ),
    evaluate_checkpoint_path(
        label="Best final retrained model (selected by validation mAP)",
        ckpt_path=BEST_FINAL_CKPT,
        loader_val=eval_val_loader,
        loader_test=eval_test_loader,
        conf_thresh=comparison_conf_thresh,
    ),
]

COMPARISON_DF = pd.DataFrame(comparison_rows)
display(COMPARISON_DF)


Experiment 10 Qualitative Error and Analysis

In [ ]:
from config import IMG_DIR

# create data loaders
train_loader, val_loader, test_loader = get_dataloaders(BATCH_SIZE, S, B, C)

# get 10 images from test set
test_ids = [test_loader.dataset.dataset.img_ids[i] for i in range(10)]

# inference threshold (higher than eval threshold)
INFERENCE_CONF_THRESH = 0.80

# create and load best model exp4
model = YOLOv1Finetune(S=S, B=B, C=C).to(DEVICE)
model.load_state_dict(torch.load(
    f"{CKPT_DIR}/exp4_Finetune_lr1e-4.pth",
    map_location=DEVICE,
    weights_only=True
))
model.eval()



# run inference on 5 test images
for img_id in test_ids:
    img_path = f"{IMG_DIR}/{img_id}.jpg"
    inference(
        model=model,
        img_path=img_path,
        S=S, B=B, C=C,
        conf_thresh=INFERENCE_CONF_THRESH,
        iou_thresh=NMS_IOU_THRESH
    )